# German cells: structure of heterogeneity

Whether a model's dispersion runs along the same social divides real Germans
disagree on, not just how much of it there is. For the same 144 German
subpopulation cells `german_adaptability.ipynb` restricts to, this notebook
builds two condensed pairwise-nEMD vectors, `D^WVS = pairwise_nemd(wvs)` and
`D^LLM = pairwise_nemd(llm)`, over the identical `names` ordering — position
`k` in one is the same pair of subpopulations as position `k` in the other,
so no square matrix or `squareform` is needed — and correlates them twice:
`ρ_structure = spearmanr(D^WVS, D^LLM)`, the headline rank reading, and
`r_structure = pearsonr(D^WVS, D^LLM)`, a linear-in-magnitude variant drawn
as its own plate beside every Spearman one. Spearman asks whether the pairs
sort the same way; Pearson also charges for the sizes of the gaps and leans
harder on the largest pairs.

`AR` (in `german_adaptability.csv`) asks how much a model disperses; `ρ` asks
whether that dispersion is aimed at the real German social axes. A model can
sit anywhere on one question independent of where it sits on the other. The
hypothesized failure mode this notebook exists to check: a marker with large
`AR` and `ρ` near zero has reacted strongly to the persona prompt without the
reaction tracking any real German cleavage — over-steerability without
social fidelity.

The reported p-values are naive: each of the `n_pairs` pairs behind a
144-cell correlation shares a cell with 142 others, so the pairs are not
independent draws — the classical Mantel-test problem — and both p-values
read as more significant than they are. Treat `ρ` and `r` as descriptive
effect sizes, not their p-values as formal tests.

A **variant** is one of the conditions a base model is evaluated under:
`base`, the untuned model with no adapter attached, or a culture LoRA such as
`german`. The CSV column and the run-directory path segment are both still
spelled `arm`, so `row["arm"]` in the code below and "variant" in this prose
name the same thing.

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = untuned `base`, solid = `german` LoRA |
| x | AR, model dispersion over survey dispersion (from the sibling CSV) |
| y | ρ (Spearman) on one plate, r (Pearson) on its variant |

Fill is the only thing that separates the two variants, so nothing is drawn
inside a marker: the white centre dot an earlier draft rode on every solid
`german` marker ate the fill at 6.5pt and made the tuned variant read as a
second hollow base, here and on the sibling plates alike.

Depends on `outputs/culture/german_adaptability.csv` already existing on
disk — run `german_adaptability.ipynb` first. Writes
`outputs/culture/german_structure.csv` and
`figures/fig_german_structure_{ntp,fa}.{png,pdf}` beside
`figures/fig_german_structure_pearson_{ntp,fa}.{png,pdf}`.

`.venv` has no jupyter and `uv run` must not be used here — it destroys the
hand-built sm_120 `llama-cpp-python` wheel. Run the code cells through
`.venv/bin/python`. Committed without cell outputs.

In [1]:
import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, spearmanr

from culture.palette import (
    MIXTRAL_ARCHIVED,
    MIXTRAL_FRESH,
    REFERENCE_MARKERS,
    REFERENCE_TONES,
    arm_order,
    model_marker,
    model_tone,
)
from culture.registry import ARMS, CULTURE_MODELS, csv_stems, is_base
from culture.tables import read_csv
from machine_bias_reproduction.analysis import load_source
from machine_bias_reproduction.config import FIGURES_ROOT, OUTPUTS_ROOT, paths_for
from machine_bias_reproduction.data import country_of
from machine_bias_reproduction.figures import save_plate
from machine_bias_reproduction.io_utils import write_csv
from machine_bias_reproduction.metrics import pairwise_nemd
from machine_bias_reproduction.plates import GRID, MUTED_INK
from machine_bias_reproduction.questions import resolve_questions

MODES = ("ntp", "fa")
MODEL_INDEX = {key: index for index, key in enumerate(CULTURE_MODELS)}
TABLE = OUTPUTS_ROOT / "culture" / "german_structure.csv"
ADAPTABILITY = OUTPUTS_ROOT / "culture" / "german_adaptability.csv"
XLABEL = "Model dispersion / survey dispersion (AR)"
METRICS = (
    ("rho_structure", "Structure of heterogeneity (ρ, Spearman)", "fig_german_structure"),
    ("pearson_structure", "Structure of heterogeneity (r, Pearson)", "fig_german_structure_pearson"),
)

## Finding the runs

Identical discovery to `german_adaptability.ipynb` — duplicated rather than
imported, since these notebooks share no code module: a missing or
incomplete run raises and is skipped, and the German cells are selected with
the same `country_of` extractor the pipeline itself groups by.

In [2]:
def sources():
    found = [
        (f"{model.label} ({arm})", f"culture/{key}/{arm}", key, model.label, arm)
        for key, model in CULTURE_MODELS.items()
        for arm in arm_order(ARMS)
    ]
    found.append((MIXTRAL_ARCHIVED, "archived", None, MIXTRAL_ARCHIVED, None))
    found.append((MIXTRAL_FRESH, "fresh", None, MIXTRAL_FRESH, None))
    return found


def load_run(source, key, arm, question):
    stems = csv_stems(key, arm, question) if key else None
    try:
        return load_source(source, paths_for(source, question.var), question, stems)
    except (FileNotFoundError, ValueError):
        return None


def german_names(prepared):
    labels = prepared.names.to_series(index=prepared.names)
    return prepared.names[(country_of(labels) == "Germany").to_numpy()]

## The structure correlation

`wvs` and `llm` are both indexed by the same `names` before either is passed
to `pairwise_nemd`, so the two condensed vectors it returns are already
pair-aligned — `spearmanr` needs nothing more than the two vectors
themselves.

In [3]:
def german_structure(prepared, mode):
    names = german_names(prepared)
    if len(names) < 4:
        return {
            "n_cells": int(len(names)),
            "n_pairs": 0,
            "rho_structure": float("nan"),
            "rho_structure_p_value": float("nan"),
            "pearson_structure": float("nan"),
            "pearson_structure_p_value": float("nan"),
        }
    columns = list(prepared.question.answer_columns)
    wvs = prepared.wvs_props.loc[names, columns].to_numpy(dtype=np.float64)
    props = prepared.ntp_props if mode == "ntp" else prepared.fa_props
    llm = props.loc[names, columns].to_numpy(dtype=np.float64)
    wvs_pairs = pairwise_nemd(wvs)
    llm_pairs = pairwise_nemd(llm)
    spearman = spearmanr(wvs_pairs, llm_pairs)
    pearson = pearsonr(wvs_pairs, llm_pairs)
    return {
        "n_cells": int(len(names)),
        "n_pairs": int(len(wvs_pairs)),
        "rho_structure": float(spearman.statistic),
        "rho_structure_p_value": float(spearman.pvalue),
        "pearson_structure": float(pearson.statistic),
        "pearson_structure_p_value": float(pearson.pvalue),
    }

## Build

One row per run and mode, same shape as the sibling notebook's table. Two
checks ride along: a complete run retains exactly 144 German cells, and
`n_pairs` always equals `n_cells * (n_cells - 1) / 2`.

In [4]:
def check(table):
    full = table[table["n_cells"] == 144]
    assert not full.empty
    assert table["n_cells"].le(144).all()
    expected_pairs = full["n_cells"] * (full["n_cells"] - 1) // 2
    assert (full["n_pairs"] == expected_pairs).all()


def build():
    rows = []
    for question in resolve_questions(None):
        for label, source, key, model_label, arm in sources():
            prepared = load_run(source, key, arm, question)
            if prepared is None:
                continue
            for mode in MODES:
                rows.append(
                    {
                        "question": question.var,
                        "question_label": question.label,
                        "model_key": key,
                        "model_label": model_label,
                        "arm": arm,
                        "series": label,
                        "source": source,
                        "mode": mode,
                        **german_structure(prepared, mode),
                    }
                )
    table = pd.DataFrame(rows)
    if not table.empty:
        check(table)
    return table

In [5]:
table = build()
write_csv(table, TABLE)
print(len(table), "rows,", int(table["n_cells"].eq(144).sum()), "complete")

74 rows, 64 complete


## Joining against the adaptability table

`ρ` alone answers half a question; the plate that matters pairs it against
`AR`, already computed in `german_adaptability.csv` — amount and direction
of the same dispersion, read together. The join key is
`["question", "source", "mode"]`, not `model_key`/`arm`: the two Mixtral
reference rows carry `NaN` in both, and pandas treats `NaN != NaN`, which
would silently drop them from an inner join on those columns.

In [6]:
adaptability = read_csv(ADAPTABILITY)
assert adaptability is not None
joined = table.merge(
    adaptability[["question", "source", "mode", "e_mean_nemd", "adaptability_ratio", "c_center_nemd"]],
    on=["question", "source", "mode"],
)
assert len(joined) == len(table)


def series_style(label, model_key, arm):
    if label in REFERENCE_TONES:
        ink = REFERENCE_TONES[label].ink
        return {
            "marker": REFERENCE_MARKERS[label],
            "color": ink,
            "markerfacecolor": ink,
            "markeredgecolor": ink if label == MIXTRAL_ARCHIVED else MUTED_INK,
            "zorder": 3.2 if label == MIXTRAL_ARCHIVED else 3.0,
        }
    index = MODEL_INDEX[model_key]
    ink = model_tone(model_key, index).ink
    return {
        "marker": model_marker(model_key, index),
        "color": ink,
        "markerfacecolor": "none" if is_base(arm) else ink,
        "markeredgecolor": ink,
        "zorder": 3.1,
    }


def structure_legend(frame):
    handles = []
    for series in dict.fromkeys(frame["series"]):
        subset = frame[frame["series"] == series]
        style = series_style(series, subset["model_key"].iat[0], subset["arm"].iat[0])
        handles.append(Line2D([], [], linestyle="none", markersize=6, label=series, **style))
    handles.append(Line2D([], [], linestyle="none", label=" "))
    handles.append(
        Line2D([], [], linestyle=(0, (4, 2)), color=MUTED_INK, label="AR = 1 / correlation = 0")
    )
    handles.append(Line2D([], [], linestyle="none", label="ideal: AR = 1 with the correlation high"))
    return handles


CORNERS = (
    (0.02, 0.97, "left", "top", "Compressed, correctly directed"),
    (0.98, 0.97, "right", "top", "Expansive, correctly directed"),
    (0.02, 0.03, "left", "bottom", "Compressed and unstructured"),
    (0.98, 0.03, "right", "bottom", "Over-steerability\nwithout social fidelity"),
)


def structure_plate(joined, mode, column, ylabel, stem):
    frame = joined[joined["mode"] == mode]
    questions = list(dict.fromkeys(joined["question"]))
    figure = Figure(figsize=(3.6 * len(questions), 4.2), layout="constrained")
    axes = figure.subplots(1, len(questions), sharey=True, squeeze=False)[0]
    right = max(1.08, float(frame["adaptability_ratio"].max()) * 1.12)
    for axis, question in zip(axes, questions, strict=True):
        subset = frame[frame["question"] == question]
        axis.axvline(1.0, linestyle=(0, (4, 2)), color=MUTED_INK, linewidth=0.9)
        axis.axhline(0.0, linestyle=(0, (4, 2)), color=MUTED_INK, linewidth=0.9)
        axis.grid(color=GRID, linewidth=0.6)
        axis.set_axisbelow(True)
        for _, row in subset.iterrows():
            style = series_style(row["series"], row["model_key"], row["arm"])
            axis.plot(
                row["adaptability_ratio"],
                row[column],
                linestyle="none",
                markersize=6.5,
                markeredgewidth=1.1,
                **style,
            )
        axis.set_title(joined.loc[joined["question"] == question, "question_label"].iat[0])
        axis.set_xlim(-0.02 * right, right)
        axis.set_ylim(-1.05, 1.05)
    axes[0].set_ylabel(ylabel)
    for x, y, ha, va, text in CORNERS:
        axes[0].text(
            x, y, text,
            transform=axes[0].transAxes,
            ha=ha, va=va,
            fontsize=7, style="italic", color=MUTED_INK,
        )
    figure.supxlabel(XLABEL, fontsize=9)
    figure.suptitle(mode.upper() if column == "rho_structure" else f"{mode.upper()} — Pearson")
    figure.legend(handles=structure_legend(frame), loc="outside right upper")
    return save_plate(figure, FIGURES_ROOT / f"{stem}_{mode}")

In [7]:
files = []
for mode in MODES:
    for column, ylabel, stem in METRICS:
        files += structure_plate(joined, mode, column, ylabel, stem)
print(len(files), "files")

8 files


## Reading the plates

- ρ near 1 in either top corner: the pairs of German subpopulations most
  different under the survey tend to be the pairs most different under the
  model, whatever the amount of dispersion.
- ρ near or below zero in the bottom-right, with AR well above 1, is the
  pattern hypothesized before any ρ existed: large dispersion aimed at no
  real social axis — over-steerability without social fidelity.
- The Pearson plate is the same square with `r` on the y axis: it agrees
  with Spearman when the relationship is roughly linear in magnitude and
  splits from it when a handful of extreme pairs carry the correlation. A
  cell whose `ρ` and `r` disagree earns a look at its pair scatter before
  either number is quoted.
- Hollow is the `base` variant, solid the `german` one, and nothing rides
  inside a marker — same encoding as the sibling notebook's plates.
- This square is independent of `german_adaptability.ipynb`'s E-AR square; a
  model can sit anywhere here regardless of where it sits there. Read the
  two side by side, not in place of each other.
- Both p-values are in the CSV, not on the plates, and should be read as
  indicative only — see the intro cell's Mantel-test caveat.
- Absent runs contribute no marker; a run with fewer than four German cells
  contributes `nan` in the CSV rather than a marker at the origin. Both axes
  are padded past their limits so a marker at `AR = 0` or `ρ = ±1` is drawn
  whole rather than clipped by the spine.